# Qwen3-VL-8B AVM Eval (Mac)

Runs **control** (text prompt only) and **few-shot** (10 image+label examples in context) on the same machine for a fair comparison.

**Before running:** open this notebook from the repo root (folder containing `TRUE_AVM_CODES` and `AVM_Reference`).

**Expected runtime on M4 Max 36GB:** ~1–2 hours per condition (~2–4 hours total).

## 1. Install dependencies (run once)

In [1]:
%pip install -q "git+https://github.com/huggingface/transformers" accelerate pillow datasets pandas


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

In [2]:
from pathlib import Path
import os

# Let MPS use more unified memory on Mac (reduces premature OOM kills)
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.0")

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
DATASET_ID = "Supermaxman/esa-hubble"
NUM_IMAGES = 266
MAX_IMAGE_EDGE = 768  # query images (was 1024; lower = less memory)
FEWSHOT_MAX_IMAGE_EDGE = 512  # demo images in few-shot prompt (11 images total)
# Ask model for brief reasoning + codes on query images (metrics still parse codes from reply)
ASK_FOR_REASONING = True
MAX_NEW_TOKENS = 256 if ASK_FOR_REASONING else 32
SEED = 42

# Append one JSON object per row with full model text (resume-safe; does not re-run finished rows)
TRACE_RAW_RESPONSES = True

# Set True to download the 8B model and run one tiny VLM forward pass on MPS (slow first time)
RUN_FULL_MPS_PROBE = False

# Toggle runs (set False to skip a condition you already finished)
RUN_CONTROL = False  # control codes file already complete
RUN_FEWSHOT = True

# 10 diverse few-shot demo indices (excluded from eval metrics for both groups)
FEWSHOT_INDICES = [
    1,   # C.5.1.7 — interacting galaxy (Local)
    3,   # D.6.2.2 — early universe
    4,   # C.3.2.1 — star (Local)
    15,  # B.4.1.3 — planetary nebula (Milky Way)
    50,  # A.2.3 — solar system
    12,  # C.5.1.1, C.5.4.6 — multi-label (Local)
    16,  # D.5.3.3, D.5.5.3 — multi-label (Early)
    57,  # B.3.1.9.1, B.3.6.1 — Milky Way star field
    74,  # C.5.1.4 — galaxy type (Local)
    163, # A.2.2, A.2.2.3.1 — multi-label (Solar System)
]
FEWSHOT_INDEX_SET = set(FEWSHOT_INDICES)
EVAL_INDICES = [i for i in range(NUM_IMAGES) if i not in FEWSHOT_INDEX_SET]

OUTPUT_DIR = Path("results_mac")
OUTPUT_DIR.mkdir(exist_ok=True)
CONTROL_PATH = OUTPUT_DIR / "control_strict_codes_v1.txt"
FEWSHOT_PATH = OUTPUT_DIR / "fewshot_10_strict_codes_v1.txt"
CONTROL_RAW_PATH = OUTPUT_DIR / "control_raw_responses.jsonl"
FEWSHOT_RAW_PATH = OUTPUT_DIR / "fewshot_raw_responses.jsonl"
COMPARISON_PATH = OUTPUT_DIR / "control_vs_fewshot_summary.txt"

QUERY_PROMPT_CODES_ONLY = "Return only the AVM code or codes for this image."
QUERY_PROMPT_WITH_REASONING = (
    "Briefly explain what astronomical objects or phenomena you see (1-3 sentences). "
    "Then on the final line, give only the AVM code or codes for this image, "
    "comma-separated if there are multiple."
)
QUERY_USER_PROMPT = QUERY_PROMPT_WITH_REASONING if ASK_FOR_REASONING else QUERY_PROMPT_CODES_ONLY

# Repo root: local AVM files
REPO_DIR = Path.cwd()
if not (REPO_DIR / "TRUE_AVM_CODES").exists():
    raise FileNotFoundError(
        "Run this notebook from the repo root (need TRUE_AVM_CODES and AVM_Reference)."
    )

print(f"Eval images (excluding {len(FEWSHOT_INDICES)} demos): {len(EVAL_INDICES)}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

Eval images (excluding 10 demos): 256
Output dir: /Users/atin5551/Documents/GitHub/ECS-189G-final-project/results_mac


## 2b. MPS compatibility check (run before loading the model)

In [3]:
import torch


def check_mps_compatibility(run_model_probe=False):
    """Quick checks for Apple Silicon MPS. Optional probe loads Qwen3-VL for one tiny inference."""
    report = {
        "pytorch_version": torch.__version__,
        "mps_built": torch.backends.mps.is_built(),
        "mps_available": torch.backends.mps.is_available(),
        "basic_ops_ok": False,
        "float16_ok": False,
        "model_probe_ok": None,
        "recommended_device": "cpu",
        "verdict": "unknown",
    }

    print(f"PyTorch: {report['pytorch_version']}")
    print(f"MPS built:   {report['mps_built']}")
    print(f"MPS available: {report['mps_available']}")

    if torch.cuda.is_available():
        report["recommended_device"] = "cuda"
        report["verdict"] = "USE_CUDA"
        print("\nCUDA detected — this notebook will use CUDA, not MPS.")
        return report

    if not report["mps_available"]:
        report["verdict"] = "MPS_UNAVAILABLE"
        print("\nFAIL: MPS not available. Use Colab or expect very slow CPU inference.")
        return report

    try:
        x = torch.randn(128, 128, device="mps")
        y = x @ x
        torch.mps.synchronize()
        report["basic_ops_ok"] = True
        print("PASS: basic MPS matmul")
    except Exception as exc:
        report["verdict"] = "MPS_BASIC_OPS_FAILED"
        print(f"\nFAIL: basic MPS ops — {exc}")
        print("Recommendation: set FORCE_DEVICE = 'cpu' below, or use Colab.")
        return report

    try:
        z = torch.ones(4, 4, device="mps", dtype=torch.float16)
        _ = z.sum()
        torch.mps.synchronize()
        report["float16_ok"] = True
        print("PASS: MPS float16 tensor ops")
    except Exception as exc:
        report["verdict"] = "MPS_FLOAT16_FAILED"
        print(f"\nWARN: MPS float16 failed — {exc}")
        print("Model load may need FORCE_DEVICE = 'cpu' or float32.")

    if run_model_probe:
        print("\nRunning full Qwen3-VL MPS probe (downloads ~17GB on first run)...")
        try:
            from PIL import Image
            from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

            probe_processor = AutoProcessor.from_pretrained(MODEL_ID)
            probe_model = Qwen3VLForConditionalGeneration.from_pretrained(
                MODEL_ID,
                dtype=torch.float16,
            ).to("mps")
            probe_model.eval()

            tiny = Image.new("RGB", (64, 64), color=(128, 64, 32))
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": tiny},
                        {"type": "text", "text": "Reply with the single word OK."},
                    ],
                }
            ]
            inputs = probe_processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )
            inputs = {k: v.to("mps") for k, v in inputs.items()}

            with torch.no_grad():
                out_ids = probe_model.generate(**inputs, max_new_tokens=8, do_sample=False)
            torch.mps.synchronize()

            trimmed = [o[len(i) :] for i, o in zip(inputs["input_ids"], out_ids)]
            reply = probe_processor.batch_decode(trimmed, skip_special_tokens=True)[0]
            report["model_probe_ok"] = True
            print(f"PASS: Qwen3-VL generate on MPS (reply snippet: {reply[:40]!r})")

            del probe_model, probe_processor, inputs, out_ids
            torch.mps.empty_cache()
        except Exception as exc:
            report["model_probe_ok"] = False
            report["verdict"] = "MPS_MODEL_PROBE_FAILED"
            report["recommended_device"] = "cpu"
            print(f"\nFAIL: Qwen3-VL probe on MPS — {type(exc).__name__}: {exc}")
            print("Recommendation: FORCE_DEVICE = 'cpu' or run on Colab.")
            return report

    if report["float16_ok"]:
        report["recommended_device"] = "mps"
        report["verdict"] = "MPS_OK"
        print("\nPASS: MPS looks usable for this notebook.")
        if not run_model_probe:
            print("Note: basic checks passed; full VLM load can still OOM. Set RUN_FULL_MPS_PROBE = True to verify.")
    else:
        report["recommended_device"] = "cpu"
        report["verdict"] = "USE_CPU"
        print("\nWARN: falling back to CPU due to float16 issues.")

    return report


MPS_CHECK = check_mps_compatibility(run_model_probe=RUN_FULL_MPS_PROBE)

# Override auto-detection if needed: None | "mps" | "cpu"
FORCE_DEVICE = None

if FORCE_DEVICE is not None:
    RECOMMENDED_DEVICE = FORCE_DEVICE
    print(f"Using FORCE_DEVICE = {FORCE_DEVICE}")
else:
    RECOMMENDED_DEVICE = MPS_CHECK["recommended_device"]

print(f"\nRecommended device for model load: {RECOMMENDED_DEVICE}")
print(f"Verdict: {MPS_CHECK['verdict']}")

PyTorch: 2.9.0
MPS built:   True
MPS available: True
PASS: basic MPS matmul
PASS: MPS float16 tensor ops

PASS: MPS looks usable for this notebook.
Note: basic checks passed; full VLM load can still OOM. Set RUN_FULL_MPS_PROBE = True to verify.

Recommended device for model load: mps
Verdict: MPS_OK


## 3. Load model (Mac: MPS / CPU)

In [4]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor


def resolve_device_and_dtype():
    device = RECOMMENDED_DEVICE
    if device == "cuda":
        return device, torch.bfloat16
    if device == "mps":
        return device, torch.float16
    return "cpu", torch.float32


DEVICE, DTYPE = resolve_device_and_dtype()
print(f"Loading model on device={DEVICE}, dtype={DTYPE}")

if MPS_CHECK["verdict"] == "MPS_MODEL_PROBE_FAILED":
    raise RuntimeError("MPS probe failed — fix device or use Colab before loading the model again.")

torch.manual_seed(SEED)

processor = AutoProcessor.from_pretrained(MODEL_ID)
try:
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    model.eval()
    print("Model loaded successfully.")
except Exception as exc:
    print(f"Model load failed on {DEVICE}: {type(exc).__name__}: {exc}")
    if DEVICE == "mps":
        print("Retrying on CPU (much slower)...")
        DEVICE, DTYPE = "cpu", torch.float32
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            dtype=DTYPE,
        ).to("cpu")
        model.eval()
        print("Model loaded on CPU.")
    else:
        raise

Loading model on device=mps, dtype=torch.float16


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Model loaded successfully.


## 4. Load dataset and labels

In [5]:
from itertools import islice
from PIL import Image
from datasets import load_dataset

Image.MAX_IMAGE_PIXELS = None  # allow very large Hubble images


def parse_true_avm_codes(path: Path):
    labels_by_row = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        row_text, codes_text = line.split(":", 1)
        row_index = int(row_text.strip())
        codes = [code.strip() for code in codes_text.split(",") if code.strip()]
        labels_by_row[row_index] = codes
    expected = list(range(len(labels_by_row)))
    if sorted(labels_by_row) != expected:
        raise ValueError("TRUE_AVM_CODES rows are missing or out of order.")
    return [labels_by_row[i] for i in expected]


avm_reference_text = (REPO_DIR / "AVM_Reference").read_text(encoding="utf-8")
TRUE_AVM_CODES = parse_true_avm_codes(REPO_DIR / "TRUE_AVM_CODES")

print(f"Loaded AVM reference: {len(avm_reference_text):,} chars")
print(f"Loaded ground truth: {len(TRUE_AVM_CODES)} rows")

dataset_stream = load_dataset(DATASET_ID, split="train", streaming=True)
rows = list(islice(dataset_stream, NUM_IMAGES))
print(f"Loaded dataset rows: {len(rows)}")

Loaded AVM reference: 7,092 chars
Loaded ground truth: 266 rows


Resolving data files:   0%|          | 0/200 [00:00<?, ?it/s]

Loaded dataset rows: 266


## 5. Prompts and inference helpers

In [6]:
import gc
import re

AVM_CODE_PATTERN = re.compile(r"\b[A-EX]\.\d+(?:\.\d+)*\b")

avm_task_instructions = """
You are an expert astronomical image classifier. Given an astronomical image, your task is to identify the type(s) of object(s) or phenomenon(a) depicted and express each as an AVM 1.1 code.

AVM 1.1 Code Format:
Each code has the form <Scale>.<Taxonomy>, where:

Scale (single letter prefix):
A = Solar System
B = Milky Way
C = Local Universe
D = Early Universe
E = Unspecified
Taxonomy (dot-separated numbers following the scale letter), e.g.:
B.4.1.3 -> Milky Way : Nebula : Type : Planetary
C.5.1.1 -> Local Universe : Galaxy : Type : Spiral
D.5.5.3 -> Early Universe : Galaxy : Grouping : Cluster

Instructions:
Carefully examine the image and identify all astronomical object types or phenomena present.
For each identified type, select the most specific matching AVM 1.1 code.
An image may contain more than one type — if so, provide one code per type, separated by commas.
Output only the AVM code(s), nothing else. No explanations, no labels, no extra text.

Output format:
Single type: B.4.1.3
Multiple types: C.5.1.1, C.5.1.2
"""

avm_system_prompt = avm_reference_text + "\n\n" + avm_task_instructions


def clear_device_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()


def prepare_image(image, max_edge=MAX_IMAGE_EDGE):
    if isinstance(image, str):
        img = Image.open(image).convert("RGB")
    elif isinstance(image, Image.Image):
        img = image.convert("RGB")
    else:
        img = Image.fromarray(image).convert("RGB")
    img.thumbnail((max_edge, max_edge), Image.Resampling.LANCZOS)
    return img


def extract_avm_codes(model_reply: str):
    return AVM_CODE_PATTERN.findall(model_reply)


def build_messages(query_image, few_shot_examples=None):
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": avm_system_prompt}],
        }
    ]

    if few_shot_examples:
        for example in few_shot_examples:
            messages.append(
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": example["image"]},
                        {
                            "type": "text",
                            "text": "Return only the AVM code or codes for this example image.",
                        },
                    ],
                }
            )
            messages.append(
                {
                    "role": "assistant",
                    "content": [
                        {
                            "type": "text",
                            "text": ", ".join(example["codes"]),
                        }
                    ],
                }
            )

    messages.append(
        {
            "role": "user",
            "content": [
                {"type": "image", "image": query_image},
                {"type": "text", "text": QUERY_USER_PROMPT},
            ],
        }
    )
    return messages


def predict_avm_response(image, few_shot_examples=None):
    """Run model once; return parsed codes and full decoded text."""
    img = prepare_image(image)
    messages = build_messages(img, few_shot_examples=few_shot_examples)

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    try:
        with torch.inference_mode():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        trimmed = [o[len(i) :] for i, o in zip(inputs["input_ids"], out_ids)]
        raw_reply = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        parsed_codes = extract_avm_codes(raw_reply)
        return {"raw_reply": raw_reply, "parsed_codes": parsed_codes}
    finally:
        del inputs
        clear_device_memory()


def predict_avm_codes(image, few_shot_examples=None):
    return predict_avm_response(image, few_shot_examples=few_shot_examples)["parsed_codes"]


few_shot_examples = [
    {
        "image": prepare_image(rows[i]["image"], max_edge=FEWSHOT_MAX_IMAGE_EDGE),
        "codes": TRUE_AVM_CODES[i],
    }
    for i in FEWSHOT_INDICES
]
print(f"Built {len(few_shot_examples)} few-shot examples")

Built 10 few-shot examples


## 6. Batch runner

In [ ]:
import json
from datetime import datetime, timezone


def load_partial_predictions(output_path: Path):
    """Load saved lines so a crashed run can resume."""
    predictions = {}
    if not output_path.exists():
        return predictions

    for line in output_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        row_text, prediction_text = line.split(":", 1)
        predictions[int(row_text.strip())] = prediction_text.strip()
    return predictions


def load_traced_row_indices(raw_path: Path):
    """Row indices that already have a JSONL trace entry."""
    traced = set()
    if not raw_path.exists():
        return traced
    for line in raw_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        traced.add(int(json.loads(line)["row_index"]))
    return traced


def append_raw_trace(raw_path: Path, record: dict):
    with raw_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_prediction_lines(predictions: dict, output_path: Path):
    lines = [f"{idx}: {predictions[idx]}" for idx in sorted(predictions)]
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def run_batch(eval_indices, output_path: Path, raw_path: Path, use_few_shot: bool):
    label = "few-shot" if use_few_shot else "control"
    predictions = load_partial_predictions(output_path)
    traced_rows = load_traced_row_indices(raw_path) if TRACE_RAW_RESPONSES else set()
    remaining = [idx for idx in eval_indices if idx not in predictions]

    if not remaining:
        print(f"Already complete ({len(predictions)} rows): {output_path}")
        return output_path

    if predictions:
        print(f"Resuming {label}: {len(predictions)} done, {len(remaining)} remaining")
    else:
        print(f"Running {label} on {len(eval_indices)} images -> {output_path}")

    if TRACE_RAW_RESPONSES:
        print(f"Raw traces -> {raw_path} (ASK_FOR_REASONING={ASK_FOR_REASONING})")

    for n, row_index in enumerate(remaining, start=1):
        print(f"[{label}] {len(predictions)+1}/{len(eval_indices)} (row {row_index})")
        examples = few_shot_examples if use_few_shot else None
        result = predict_avm_response(rows[row_index]["image"], few_shot_examples=examples)
        predicted_codes = result["parsed_codes"]
        code_text = ", ".join(predicted_codes) if predicted_codes else ""
        predictions[row_index] = code_text
        save_prediction_lines(predictions, output_path)

        if TRACE_RAW_RESPONSES and row_index not in traced_rows:
            append_raw_trace(
                raw_path,
                {
                    "row_index": row_index,
                    "condition": label,
                    "use_few_shot": use_few_shot,
                    "ask_for_reasoning": ASK_FOR_REASONING,
                    "query_prompt": QUERY_USER_PROMPT,
                    "raw_reply": result["raw_reply"],
                    "parsed_codes": predicted_codes,
                    "true_codes": TRUE_AVM_CODES[row_index],
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                },
            )
            traced_rows.add(row_index)

    print(f"Saved {len(predictions)} predictions to {output_path}")
    if TRACE_RAW_RESPONSES:
        print(f"Raw traces: {len(traced_rows)} rows in {raw_path}")
    return output_path

: 

## 7. Run control + few-shot

In [ ]:
if RUN_CONTROL:
    run_batch(EVAL_INDICES, CONTROL_PATH, CONTROL_RAW_PATH, use_few_shot=False)
else:
    print("Skipped control run")


Resuming control: 10 done, 246 remaining
[control] 11/256 (row 14)
[control] 12/256 (row 17)
[control] 13/256 (row 18)
[control] 14/256 (row 19)
[control] 15/256 (row 20)
[control] 16/256 (row 21)
[control] 17/256 (row 22)
[control] 18/256 (row 23)
[control] 19/256 (row 24)
[control] 20/256 (row 25)
[control] 21/256 (row 26)
[control] 22/256 (row 27)
[control] 23/256 (row 28)
[control] 24/256 (row 29)
[control] 25/256 (row 30)
[control] 26/256 (row 31)
[control] 27/256 (row 32)
[control] 28/256 (row 33)
[control] 29/256 (row 34)
[control] 30/256 (row 35)
[control] 31/256 (row 36)
[control] 32/256 (row 37)
[control] 33/256 (row 38)
[control] 34/256 (row 39)
[control] 35/256 (row 40)
[control] 36/256 (row 41)
[control] 37/256 (row 42)
[control] 38/256 (row 43)
[control] 39/256 (row 44)
[control] 40/256 (row 45)
[control] 41/256 (row 46)
[control] 42/256 (row 47)
[control] 43/256 (row 48)
[control] 44/256 (row 49)
[control] 45/256 (row 51)
[control] 46/256 (row 52)
[control] 47/256 (row 5

In [ ]:
if RUN_FEWSHOT:
    run_batch(EVAL_INDICES, FEWSHOT_PATH, FEWSHOT_RAW_PATH, use_few_shot=True)
else:
    print("Skipped few-shot run")

Resuming few-shot: 55 done, 201 remaining
Raw traces -> results_mac/fewshot_raw_responses.jsonl (ASK_FOR_REASONING=True)
[few-shot] 56/256 (row 63)


## 8. Metrics and comparison

In [ ]:
import pandas as pd


def parse_prediction_file(path: Path):
    predictions = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        row_text, prediction_text = line.split(":", 1)
        row_index = int(row_text.strip())
        predictions[row_index] = AVM_CODE_PATTERN.findall(prediction_text)
    return predictions


def precision_recall_f1(predicted_codes, true_codes):
    predicted_set = set(predicted_codes)
    true_set = set(true_codes)
    if not predicted_set and not true_set:
        return 1.0, 1.0, 1.0
    if not predicted_set or not true_set:
        return 0.0, 0.0, 0.0
    correct = len(predicted_set & true_set)
    precision = correct / len(predicted_set)
    recall = correct / len(true_set)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def code_ancestors(code):
    parts = code.split(".")
    return [".".join(parts[:end]) for end in range(1, len(parts) + 1)]


def expand_with_ancestors(codes):
    expanded = set()
    for code in codes:
        expanded.update(code_ancestors(code))
    return expanded


def compute_metrics(predictions: dict, eval_indices):
    metric_rows = []
    for row_index in eval_indices:
        true_codes = TRUE_AVM_CODES[row_index]
        predicted_codes = predictions.get(row_index, [])
        precision, recall, f1 = precision_recall_f1(predicted_codes, true_codes)
        h_precision, h_recall, h_f1 = precision_recall_f1(
            expand_with_ancestors(predicted_codes),
            expand_with_ancestors(true_codes),
        )
        metric_rows.append(
            {
                "row_index": row_index,
                "predicted_codes": predicted_codes,
                "true_codes": true_codes,
                "exact_set_match": set(predicted_codes) == set(true_codes),
                "over_predicted": len(predicted_codes) > len(true_codes),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "hierarchical_f1": h_f1,
            }
        )
    df = pd.DataFrame(metric_rows)
    summary = {
        "rows_evaluated": len(df),
        "exact_set_matches": int(df["exact_set_match"].sum()),
        "exact_match_rate": float(df["exact_set_match"].mean()),
        "macro_f1": float(df["f1"].mean()),
        "macro_hierarchical_f1": float(df["hierarchical_f1"].mean()),
        "over_prediction_rows": int(df["over_predicted"].sum()),
    }
    return df, summary


def print_summary(name, summary):
    print(f"\n=== {name} ===")
    print(f"Exact matches: {summary['exact_set_matches']} / {summary['rows_evaluated']}")
    print(f"Exact match rate: {summary['exact_match_rate']:.4f}")
    print(f"Macro F1: {summary['macro_f1']:.4f}")
    print(f"Hierarchical F1: {summary['macro_hierarchical_f1']:.4f}")
    print(f"Over-prediction rows: {summary['over_prediction_rows']}")


if not CONTROL_PATH.exists() or not FEWSHOT_PATH.exists():
    raise FileNotFoundError("Run section 7 first so both prediction files exist.")

control_preds = parse_prediction_file(CONTROL_PATH)
fewshot_preds = parse_prediction_file(FEWSHOT_PATH)

control_df, control_summary = compute_metrics(control_preds, EVAL_INDICES)
fewshot_df, fewshot_summary = compute_metrics(fewshot_preds, EVAL_INDICES)

print_summary("CONTROL (text prompt only)", control_summary)
print_summary("FEW-SHOT (10 image+label examples)", fewshot_summary)

delta = {
    "exact_match_rate_delta": fewshot_summary["exact_match_rate"] - control_summary["exact_match_rate"],
    "macro_f1_delta": fewshot_summary["macro_f1"] - control_summary["macro_f1"],
    "hierarchical_f1_delta": fewshot_summary["macro_hierarchical_f1"] - control_summary["macro_hierarchical_f1"],
}
print("\n=== DELTA (few-shot minus control) ===")
for key, value in delta.items():
    print(f"{key}: {value:+.4f}")

comparison_lines = [
    "CONTROL vs FEW-SHOT (Mac run)",
    f"Eval images: {len(EVAL_INDICES)} (excluded demo indices: {sorted(FEWSHOT_INDICES)})",
    "",
    f"Control exact: {control_summary['exact_set_matches']}/{control_summary['rows_evaluated']} ({control_summary['exact_match_rate']:.4f})",
    f"Few-shot exact: {fewshot_summary['exact_set_matches']}/{fewshot_summary['rows_evaluated']} ({fewshot_summary['exact_match_rate']:.4f})",
    f"Control macro F1: {control_summary['macro_f1']:.4f}",
    f"Few-shot macro F1: {fewshot_summary['macro_f1']:.4f}",
    f"Control hierarchical F1: {control_summary['macro_hierarchical_f1']:.4f}",
    f"Few-shot hierarchical F1: {fewshot_summary['macro_hierarchical_f1']:.4f}",
    "",
    f"Delta exact match rate: {delta['exact_match_rate_delta']:+.4f}",
    f"Delta macro F1: {delta['macro_f1_delta']:+.4f}",
    f"Delta hierarchical F1: {delta['hierarchical_f1_delta']:+.4f}",
]
COMPARISON_PATH.write_text("\n".join(comparison_lines) + "\n", encoding="utf-8")
print(f"\nWrote summary to {COMPARISON_PATH.resolve()}")

control_df.to_csv(OUTPUT_DIR / "control_metrics.csv", index=False)
fewshot_df.to_csv(OUTPUT_DIR / "fewshot_metrics.csv", index=False)
print(f"Wrote per-row CSVs to {OUTPUT_DIR.resolve()}")

## 9. Quick error peek (rows where few-shot helped / hurt)

In [ ]:
merged = control_df.merge(
    fewshot_df,
    on="row_index",
    suffixes=("_control", "_fewshot"),
)

merged["fewshot_better"] = (
    merged["f1_fewshot"] > merged["f1_control"]
) | (
    merged["exact_set_match_fewshot"] & ~merged["exact_set_match_control"]
)
merged["fewshot_worse"] = (
    merged["f1_fewshot"] < merged["f1_control"]
) | (
    merged["exact_set_match_control"] & ~merged["exact_set_match_fewshot"]
)

print(f"Few-shot better on {merged['fewshot_better'].sum()} rows")
print(f"Few-shot worse on {merged['fewshot_worse'].sum()} rows")
print(f"Both wrong (exact): {(~merged['exact_set_match_control'] & ~merged['exact_set_match_fewshot']).sum()} rows")

print("\nSample improvements:")
display(
    merged.loc[merged["fewshot_better"], [
        "row_index", "true_codes_control", "predicted_codes_control", "predicted_codes_fewshot"
    ]].head(5)
)

print("\nSample regressions:")
display(
    merged.loc[merged["fewshot_worse"], [
        "row_index", "true_codes_control", "predicted_codes_control", "predicted_codes_fewshot"
    ]].head(5)
)